In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

# os.environ["ASTRA_DB_API_ENDPOINT"] = os.getenv("ASTRA_DB_API_ENDPOINT")
# os.environ["ASTRA_DB_APPLICATION_TOKEN"] = os.getenv("ASTRA_DB_APPLICATION_TOKEN")
# os.environ["Groq_API_KEY"] = getpass("Enter your Groq API Key: ")

In [ ]:
from langchain_astradb import AstraDBVectorStore
from langchain_community.embeddings import OllamaEmbeddings

# print("TOKEN:", os.getenv("ASTRA_DB_APPLICATION_TOKEN"))
# print("ENDPOINT:", os.getenv("ASTRA_DB_API_ENDPOINT"))

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vstore = AstraDBVectorStore(
    collection_name="test_collection2",
    embedding=embeddings,
    token=os.getenv("ASTRA_DB_APPLICATION_TOKEN"),
    api_endpoint=os.getenv("ASTRA_DB_API_ENDPOINT")
)
print("Astra vector configured successfully")

In [ ]:
from datasets import load_dataset # Imports a function from Hugging Face Datasets

dataset = load_dataset("datastax/philosopher-quotes")["train"]
# 👉 This will:
    ## download dataset
    ## load it into Python
    ## give structured data (train/test splits)
print("example: ", dataset[1])

In [ ]:
from langchain.schema import Document
docs = []
for data in dataset:
    metadata = {"author" : data["author"]}
    if data["tags"]:
        for tag in data["tags"].split(";"):
            metadata[tag] = "y"
    doc = Document(page_content=data["quote"], metadata=metadata)
    docs.append(doc)
print(docs)
            

In [ ]:
vector_embeddings = vstore.add_documents(docs)
print(f"Documents inserted : {len(vector_embeddings)}")

In [ ]:
# Check your colllection to verify that the doc are embedded
print(vstore.astra_db.collection("test_collection2").find())

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain_community.llms import Ollama
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnablePassthrough
# from langchain_groq import ChatGroq

llm = Ollama(model="llama3.2")
# llm = ChatGroq(groq_api_key = os.getenv("Qroq_API_KEY") , model_name="meta-llama/llama-4-scout-17b-16e-instruct")
retriever = vstore.as_retriever(search_kwargs={"k" : 3})

prompt_template = """
Answer the question based only on the supplied context. If you don't know the answer, say you don't know the answer.
Context: {context}
Question: {question}
Your answer:
"""

prompt = ChatPromptTemplate.from_template(prompt_template)

chain = (
    {"context" : retriever, "question" : RunnablePassthrough()} | prompt | llm | StrOutputParser()
)

In [ ]:
chain.invoke("who qouted this \"The proof that you know something is that you are able to teach it\", and what are the tags ")

In [ ]:
# # WARNING: This will delete the collection and all documents in the collection
# vstore.delete_collection()